# El Estado colombiano ya contrata inteligencia artificial: quién, qué y cuánto

Análisis del [newsletter de Croma](https://usecroma.substack.com), número 2 (noviembre de 2026). Todo sale de fuentes oficiales consultadas con la [API de Croma](https://usecroma.com):

- **SECOP I y SECOP II**, todos los contratos publicados, vía `/co/secop/contracts-search/v1`.
- **Ley 996 de 2005** (Ley de Garantías), artículo 33, vía `/co/funcion-publica/norm/v1`.

**La pregunta.** ¿Cuántos contratos públicos mencionan la inteligencia artificial, quién los firma, qué se compra y cuánto se paga?

**Cómo correrlo.** Con `CROMA_API_KEY` en el entorno, el notebook consulta la API y actualiza `data/`. Sin API Key, lee `data/contratos-ia.csv`, la foto de los datos usada en el artículo (consultada el 4 de noviembre de 2026).

**Privacidad.** Cuando el contratista es una persona natural, el CSV no guarda su nombre ni su documento. Tampoco guarda supervisores ni ordenadores del gasto.

In [1]:
import re
import sys
import unicodedata
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))
import croma

DATA = Path("data")
CHARTS = Path("charts")
DATA.mkdir(exist_ok=True)
CHARTS.mkdir(exist_ok=True)
ONLINE = croma.has_key()
print("Consultando la API de Croma" if ONLINE else "Sin CROMA_API_KEY: leyendo data/contratos-ia.csv")

pd.set_option("display.max_colwidth", 110)
MM = 1e6  # millones de pesos
YEARS = [2022, 2023, 2024, 2025, 2026]

Consultando la API de Croma


## Los datos

Buscamos `"inteligencia artificial"` año por año (por fecha de firma) y bajamos todas las páginas. La búsqueda es de texto completo sobre el objeto del contrato y las partes, así que también trae coincidencias que no son de IA. Por eso la depuramos en el paso siguiente.

In [2]:
NATURAL = {"Cédula de Ciudadanía", "Cédula de Extranjería", "Nit de Persona Natural", "Registro Civil"}
KEEP = ["id", "platform", "contract_id", "year", "sign_date", "entity", "entity_nit", "entity_order", "entity_sector",
        "entity_department", "provider", "provider_document", "provider_document_type", "object", "description",
        "contract_type", "modality", "status", "value", "url"]
CEDULA = re.compile(r"(?i)\b(c\.?\s?c\.?|c[eé]dula)\s*(n[oº°.]*)?\s*[\d.]{6,13}")


def fetch() -> pd.DataFrame:
    path = DATA / "contratos-ia.csv"
    if ONLINE:
        rows = []
        for year in YEARS:
            rows += croma.search_all("/co/secop/contracts-search/v1", max_pages=20, query="inteligencia artificial",
                                     year=year, per_page=100, sort="value_desc")
        df = pd.DataFrame(rows)[KEEP]
        person = df.provider_document_type.isin(NATURAL)
        df.loc[person, ["provider", "provider_document"]] = None  # personas naturales: sin nombre ni documento
        for col in ["object", "description"]:
            df[col] = df[col].fillna("").str.replace(CEDULA, "[documento]", regex=True)
        df.to_csv(path, index=False)
    return pd.read_csv(path, dtype={"entity_nit": str, "provider_document": str, "contract_id": str})


raw = fetch()
raw.groupby("year").size().rename("resultados de la búsqueda")

year
2022     96
2023    137
2024    267
2025    611
2026    569
Name: resultados de la búsqueda, dtype: int64

## Depuración

1. **Frase exacta.** Nos quedamos con los contratos cuyo objeto o descripción dice "inteligencia artificial" (sin importar tildes, mayúsculas o saltos de línea). Se caen los que coincidieron por otra cosa, como un contratista que se llama "Unión Temporal Inteligencia Artificial" o "fibras artificiales".
2. **Duplicados.** Un mismo contrato publicado dos veces con el número escrito distinto (un espacio de más) cuenta una vez.
3. **Quién firma.** Las cámaras de comercio publican en SECOP, pero son entidades privadas. Las separamos del resto para no contarlas como Estado.

In [3]:
def norm(text: str) -> str:
    text = unicodedata.normalize("NFKD", str(text or "")).encode("ascii", "ignore").decode().lower()
    return re.sub(r"\s+", " ", text)


raw["texto"] = (raw.object.fillna("") + " " + raw.description.fillna("")).map(norm)
ia = raw[raw.texto.str.contains(r"inteligencia\s*artificial")].copy()
print("Sin la frase exacta:", len(raw) - len(ia))
print(raw[~raw.index.isin(ia.index)].object.map(norm).str[:110].to_string())

ia["contract_key"] = ia.contract_id.fillna("").str.replace(r"\s+", " ", regex=True)
before = len(ia)
ia = ia.drop_duplicates(["platform", "contract_key", "entity_nit", "value"])
print("\nDuplicados:", before - len(ia))

ia["camara"] = ia.entity.map(norm).str.contains("camara de comercio")
ia["persona_natural"] = ia.provider_document_type.isin(NATURAL)
ia["fecha"] = pd.to_datetime(ia.sign_date)
ia["firmante"] = ia.camara.map({True: "Cámaras de comercio", False: "Entidades públicas"})
print("Contratos que quedan:", len(ia))

Sin la frase exacta: 11
206     adquisicion de otros productos quimicos; fibras artificiales; productos farmaceuticos; jabones; champu; en...
425     adquisicion de software profesional por 1 ano de licenciamiento para la oficina asesora de planeacion estr...
512     desarrollar el proyecto de innovacion multicampus ladera cali; mediante el suministro; instalacion y dotac...
536     adelantar proyectos de investigacion y desarrollo tecnologico e innovacion; con desarrollo de nuevos proce...
571           suministrar licenciamientos de software para laboratorios de mac; multimedia; diseno de software libre.
636     prestar el servicio tecnologico de implementacion y operacion de una mesa de ayuda omnicanal para la gesti...
888     adquisicion de software profesional por 1 ano de licenciamiento para la oficina asesora de planeacion estr...
1061    contratar la adquisicion de productos quimicos; fibras artificiales y otros relacionados para ser comercia...
1122       suministro; dotacion 

## 1. Cuántos contratos por año

2026 llega hasta el 3 de noviembre. Para comparar con 2025 sin ventaja, contamos también enero a octubre de cada año.

In [4]:
by_year = ia.pivot_table(index="year", columns="firmante", values="id", aggfunc="count", fill_value=0)
by_year["Total"] = by_year.sum(axis=1)
by_year

firmante,Cámaras de comercio,Entidades públicas,Total
year,,,
2022,1,95,96
2023,31,104,135
2024,57,208,265
2025,183,421,604
2026,192,372,564


In [5]:
jan_oct = ia[ia.fecha.dt.month <= 10]
same_period = jan_oct.pivot_table(index="year", columns="firmante", values="id", aggfunc="count", fill_value=0)
same_period["Total"] = same_period.sum(axis=1)
same_period["variación total"] = same_period.Total.pct_change().mul(100).round(0)
same_period["variación públicas"] = same_period["Entidades públicas"].pct_change().mul(100).round(0)
same_period

firmante,Cámaras de comercio,Entidades públicas,Total,variación total,variación públicas
year,,,,,
2022,0,88,88,NaN,NaN
2023,24,91,115,31.0,3.0
2024,44,169,213,85.0,86.0
2025,137,343,480,125.0,103.0
2026,192,372,564,18.0,8.0


## 2. Enero de 2026: la carrera antes de la Ley de Garantías

El artículo 33 de la Ley 996 de 2005 prohíbe la contratación directa de todas las entidades del Estado durante los cuatro meses anteriores a la elección presidencial y hasta la segunda vuelta. En 2026 eso fue del 31 de enero al 21 de junio. Lo leemos de la fuente:

In [6]:
def article_33() -> str:
    path = DATA / "ley-996-2005-articulo-33.txt"
    if ONLINE:
        norm_text = croma.call("/co/funcion-publica/norm/v1", norm_id="18232", offset=30000, limit=30000)["norm"]["content"]["text"]
        start = norm_text.find("Artículo 33.")
        path.write_text(norm_text[start:norm_text.find("Artículo 34.", start)].strip())
    return path.read_text()


print(article_33()[:420])

Artículo 33. Restricciones a la contratación pública. Durante los cuatro (4) meses anteriores a la elección presidencial y hasta la realización de la elección en la segunda vuelta, si fuere el caso, queda prohibida la contratación directa por parte de todos los entes del Estado.

Queda exceptuado lo referente a la defensa y seguridad del Estado, los contratos de crédito público, los requeridos para cubrir las emergen


In [7]:
public = ia[~ia.camara]
monthly = public.assign(mes=public.fecha.dt.to_period("M")).groupby("mes").agg(
    contratos=("id", "count"),
    personas_naturales=("persona_natural", "sum"),
    directa=("modality", lambda m: m.fillna("").str.contains("irecta").sum()),
).reindex(pd.period_range("2022-01", "2026-10", freq="M"), fill_value=0)  # los meses sin contratos cuentan como 0
monthly.loc["2025-01":]

,contratos,personas_naturales,directa
2025-01,31,27,18
2025-02,53,48,48
2025-03,38,32,34
2025-04,24,10,16
2025-05,36,28,33
2025-06,25,15,21
2025-07,16,8,13
2025-08,45,29,38
2025-09,38,24,25
2025-10,37,20,25


In [8]:
jan = public[(public.fecha >= "2026-01-01") & (public.fecha <= "2026-01-31")]
print(f"Enero de 2026, entidades públicas: {len(jan)} contratos, {jan.persona_natural.sum()} con personas naturales")
print(f"Firmados del 26 al 30 de enero: {(jan.fecha >= '2026-01-26').sum()}")
window = public[(public.fecha >= "2026-01-31") & (public.fecha <= "2026-06-21")]
print(f"31 de enero a 21 de junio: {len(window)} contratos en {window.fecha.dt.to_period('M').nunique()} meses")
print("Modalidad dentro de la ventana:")
print(window.modality.value_counts().head(6).to_string())
print(f"Enero como parte de enero-octubre de 2026: {len(jan) / len(public[(public.year == 2026) & (public.fecha.dt.month <= 10)]):.0%}")

Enero de 2026, entidades públicas: 220 contratos, 201 con personas naturales
Firmados del 26 al 30 de enero: 79
31 de enero a 21 de junio: 10 contratos en 5 meses
Modalidad dentro de la ventana:
modality
Contratación régimen especial                  4
Régimen Especial                               2
Contratación directa                           2
Contratación régimen especial (con ofertas)    1
Mínima cuantía                                 1
Enero como parte de enero-octubre de 2026: 59%


Pasada la veda, ¿volvió el ritmo de 2025? Julio a octubre de cada año:

In [9]:
jul_oct = public[public.fecha.dt.month.between(7, 10)].groupby("year").size()
jul_oct.loc[[2024, 2025, 2026]].rename("contratos julio-octubre")

year
2024     78
2025    136
2026    139
Name: contratos julio-octubre, dtype: int64

## 3. Qué se contrata

**Quién lo hace.** Primero, si el contratista es una persona natural (un profesional o un instructor) o una organización.

In [10]:
people = public.groupby("year").persona_natural.agg(["sum", "count"])
people["% personas naturales"] = (people["sum"] / people["count"] * 100).round(0)
people.rename(columns={"sum": "con personas naturales", "count": "contratos"})

,con personas naturales,contratos,% personas naturales
year,,,
2022,67,95,71.0
2023,76,104,73.0
2024,136,208,65.0
2025,261,421,62.0
2026,280,372,75.0


**Qué dice el objeto.** Clasificamos cada contrato por las palabras de su objeto, en este orden: la primera regla que coincide gana. Las reglas son literales a propósito, para que cualquiera las pueda revisar y discutir.

- **Formación y eventos**: curso, capacitación, formación, diplomado, taller, charla, instructor, docente, evento, foro...
- **Licencias y suscripciones**: licencia, suscripción, SaaS, ChatGPT, Copilot...
- **Acompañamiento a empresas**: empresas, empresarios, emprendedores, mipymes, clúster...
- **Desarrollo y análisis de datos**: desarrollo, implementación, diseño, chatbot, analítica, datos, plataforma, investigación...
- **Otro**: lo que no coincide con ninguna.

In [11]:
CATEGORIES = [
    ("Formación y eventos", r"capacit|\bformacion|\bformar\b|\bcurso|diplomad|taller|workshop|charla|conferenci|seminario"
                            r"|instructor|facilitador|docente|ensenanza|\bclases?\b|\bevento|encuentro|\bforo\b|congreso"
                            r"|bootcamp|tecnoacademia|estudiante|alfabetizacion|transferencia de conocimiento"),
    ("Licencias y suscripciones", r"licenci|suscripci|\bsaas\b|chatgpt|\bcopilot\b|workspace"),
    ("Acompañamiento a empresas", r"\bempresas\b|empresari|emprende|mipyme|cluster|comerciantes|tejido empresarial"),
    ("Desarrollo y análisis de datos", r"desarroll|implement|diseno|chatbot|asistente virtual|prototipo|automatiz|analitic"
                                       r"|solucion|plataforma|software|\bmodelos?\b|herramienta|\bdatos\b|investigacion|ciencia"),
]


def categorize(text: str) -> str:
    for name, pattern in CATEGORIES:
        if re.search(pattern, text):
            return name
    return "Otro"


ia["categoria"] = ia.object.map(norm).map(categorize)
public = ia[~ia.camara]
recent = public[public.year >= 2025]
mix = pd.crosstab(recent.categoria, recent.persona_natural.map({True: "persona natural", False: "organización"}))
mix["total"] = mix.sum(axis=1)
mix["%"] = (mix.total / mix.total.sum() * 100).round(0)
mix.sort_values("total", ascending=False)

persona_natural,organización,persona natural,total,%
categoria,,,,
Desarrollo y análisis de datos,100,367,467,59.0
Formación y eventos,90,114,204,26.0
Otro,24,36,60,8.0
Licencias y suscripciones,31,18,49,6.0
Acompañamiento a empresas,7,6,13,2.0


Una muestra al azar de cada categoría, para juzgar la regla a ojo:

In [12]:
for name, group in recent.groupby("categoria"):
    print(f"\n== {name} ({len(group)})")
    print(group.sample(min(6, len(group)), random_state=7).object.map(norm).str[:130].to_string(index=False))


== Acompañamiento a empresas (13)
prestar los servicios para incorporar; desarrollar e implementar el modelo de arquitectura empresarial (mae...
prestacion de servicios tecnologicos para la implementacion; operacion; soporte; despliegue y administracio...
prestar servicios profesionales en la implementacion del modelo de arquitectura empresarial; el gobierno de...
prestar servicios profesionales como investigador experto para el desarrollo de proyectos de investigacion;...
33300062604 - implementar y ejecutar una estrategia de alcance nacional para fomentar el uso y adopcion de ...
contratar la prestacion de los servicios para fortalecer empresas de alto impacto de bogota; en tematicas r...

== Desarrollo y análisis de datos (467)
realizar la supervision tecnica independiente de conformidad con lo establecido en la nsr-10 de los element...
prestar servicios profesionales en los procesos de innovacion y analitica de datos de la direccion general ...
implementar modelo de inteligencia a

**Las marcas.** Cuántos contratos (de todos, incluidas las cámaras) nombran una herramienta concreta:

In [13]:
BRANDS = {"ChatGPT u OpenAI": r"chatgpt|openai", "Copilot": r"\bcopilot\b", "Gemini": r"\bgemini\b", "Claude": r"\bclaude\b"}
brands = pd.DataFrame([{"herramienta": b, "contratos": ia.texto.str.contains(p).sum()} for b, p in BRANDS.items()])
print(brands.to_string(index=False))
ia[ia.texto.str.contains(r"chatgpt|openai|\bcopilot\b|\bgemini\b|\bclaude\b") & ~ia.persona_natural][["year", "entity", "value", "object"]].assign(
    object=lambda d: d.object.map(norm).str[:100]).sort_values("year")

     herramienta  contratos
ChatGPT u OpenAI         10
         Copilot          3
          Gemini          2
          Claude          2


,year,entity,value,object
207,2023,CALDAS - CÁMARA DE COMERCIO DE MANIZALES POR CALDAS,4760000.00,realizar la formacion especializada de 5 horas de manera presencial y de forma teorico practica; den
431,2024,SANTANDER - CAMARA DE COMERCIO DE BUCARAMANGA,10710000.00,desarrollar nuevas mejoras y actualizaciones de la plataforma de inteligencia artificial; que permit
455,2024,IGAC Sede Central *,4514100.00,contratar la suscripcion de chatgpt team para potenciar la colaboracion y desarrollo de proyectos de
784,2025,FONDO DE DESARROLLO LOCAL DE LA CANDELARIA,36231020.00,prestar servicios para la implementacion y acompanamiento de estrategias orientadas a la apropiacion
889,2025,CORPORACION PARA EL DESARROLLO SOSTENIBLE DEL URABA,14634112.00,contratar los servicios de educacion especializada para la formacion y el trabajo para capacitar al
951,2025,IGAC Sede Central *,7436086.00,contratar la suscripcion de chatgpt team para potenciar la colaboracion y desarrollo de proyectos de
1040,2025,RISARALDA - CÁMARA DE COMERCIO DE PEREIRA,2300000.00,solicitud por concepto de honorarios para la empresa quinoa sas para capacitar a empresarios en el u
1091,2025,BOGOTÁ - CÁMARA DE COMERCIO DE BOGOTÁ,39270.00,renovacion y adquisicion de licencias del modelo de inteligencia artificial conocido como chatgpt 4.
1470,2026,MUNICIPIO DE ANDES,20566930.01,actualizacion de 36 licenciamientos workspace enterprise starter de las cuentas de correos electroni
1487,2026,CASANARE - CAMARA DE COMERCIO,17900000.00,compraventa de licencia para el servicio saas atera itd- expert acceso full; para la gestion de la m


## 4. Quién contrata

Las entidades públicas con más contratos en 2025 y 2026, y si son nacionales o territoriales.

In [14]:
top_entities = recent.groupby("entity").agg(contratos=("id", "count"), millones=("value", lambda v: round(v.sum() / MM))).sort_values(
    "contratos", ascending=False).head(12)
top_entities

,contratos,millones
entity,,
INSTITUCIÓN UNIVERSITARIA ITM,51,1690
JEP,37,4205
FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES,22,182336
SERVICIO GEOLOGICO COLOMBIANO**,21,1854
IGAC Sede Central *,19,1241
"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, LA CIENCIA Y LA TECNOLOGÍA, ATENEA",18,17707
TRANSMILENIO S.A.,17,1733
UNP,16,844
FONDO NACIONAL AMBIENTAL - ANLA,16,1171


In [15]:
order = recent.entity_order.fillna("sin dato").str.upper().str.extract(r"(NACIONAL|TERRITORIAL|DISTRIT|CORPORACI)")[0].fillna("SIN DATO")
order.value_counts()

0
NACIONAL       483
TERRITORIAL    295
CORPORACI       15
Name: count, dtype: int64

In [16]:
camaras = ia[ia.camara & (ia.year >= 2025)]
print(f"Cámaras de comercio en 2025-2026: {len(camaras)} contratos de {len(ia[ia.year >= 2025])}")
camaras.entity.value_counts().head(5)

Cámaras de comercio en 2025-2026: 375 contratos de 1168


entity
ATLÁNTICO - CÁMARA DE COMERCIO DE BARRANQUILLA         77
CALDAS - CÁMARA DE COMERCIO DE MANIZALES POR CALDAS    31
RISARALDA - CÁMARA DE COMERCIO DE PEREIRA              25
BOGOTÁ - CÁMARA DE COMERCIO DE BOGOTÁ                  19
SANTANDER - CAMARA DE COMERCIO DE BUCARAMANGA          19
Name: count, dtype: int64

Qué contratan las cámaras en 2026, con las mismas reglas de clasificación:

In [17]:
ia[ia.camara & (ia.year == 2026)].categoria.value_counts(normalize=True).mul(100).round(0)

categoria
Formación y eventos               59.0
Acompañamiento a empresas         22.0
Desarrollo y análisis de datos    11.0
Otro                               4.0
Licencias y suscripciones          4.0
Name: proportion, dtype: float64

## 5. Cuánta plata

El valor de un contrato que menciona la IA no es gasto en IA: muchos contratos grandes la nombran como una línea entre varias. Por eso miramos la mediana, que no se mueve con un contrato gigante, y aparte la concentración.

In [18]:
values = public.groupby("year").value.agg(contratos="count", total=lambda v: v.sum() / MM, mediana=lambda v: v.median() / MM).round(1)
values

,contratos,total,mediana
year,,,
2022,95,28456.1,41.2
2023,104,22998.5,42.2
2024,208,503555.1,40.5
2025,421,256413.1,45.8
2026,372,88767.0,62.5


In [19]:
fut = public[public.entity.map(norm).str.contains("fondo unico de tecnologias")]
share = fut.groupby("year").value.sum() / public.groupby("year").value.sum() * 100
print("Participación del Fondo Único de TIC (MinTIC) en el valor total, por año (%):")
print(share.round(0).fillna(0).to_string())

Participación del Fondo Único de TIC (MinTIC) en el valor total, por año (%):
year
2022     1.0
2023    21.0
2024    87.0
2025    67.0
2026    12.0


In [20]:
orgs = public[~public.persona_natural & (public.year == 2026)]
orgs.nlargest(8, "value").assign(millones=lambda d: (d.value / MM).round(0), object=lambda d: d.object.map(norm).str[:120])[
    ["sign_date", "entity", "provider", "millones", "object"]]

,sign_date,entity,provider,millones,object
1111,2026-01-30,"AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR, LA CIENCIA Y LA TECNOLOGÍA, ATENEA",Universidad de los Andes,15672.0,otorgar apoyo economico a la universidad de los andes en la modalidad de recuperacion contingente para la ...
1112,2026-06-19,BOGOTÁ D.C. - PATRIMONIOS AUTONOMOS FIDUCIARIA COLPATRIA S.A.,UNIoN TEMPORAL DATA 2026,15097.0,realizar la etapa de preingenieria; construccion; dotacion; comisionamiento; urbanismo; datacenter y puest...
1113,2026-01-29,FONDO UNICO DE TECNOLOGÍAS DE LA INFORMACIÓN Y LAS COMUNICACIONES,UdeC (Proveedor),9620.0,3200042 - prestar servicios de asistencia tecnica especializada para implementar las acciones del conpes 4...
1114,2026-07-03,BOGOTÁ D.C. - PATRIMONIOS AUTONOMOS FIDUCIARIA COLPATRIA S.A.,ARQ S.A.S.,4224.0,interventoria integral para el seguimiento tecnico; administrativo; financiero; contable; tributario y jur...
1115,2026-10-15,AGENCIA DISTRITAL PARA LA EDUCACIÓN SUPERIOR; LA CIENCIA Y LA TECNOLOGÍA; ATENEA,KUEPA EDUTECH SAS,3239.0,contratar el operador para la implementacion de la estrategia del programa talento capital formacion para ...
1116,2026-04-28,INSTITUTO TECNICO NACIONAL DE COMERCIO SIMON RODRIGUEZ,CORPORACION ENLACES,2600.0,aunar esfuerzos tecnicos; administrativos y financieros para fortalecer los procesos de investigacion de l...
1117,2026-09-03,DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA,FUNDACION POR UNA SONRISA,2429.0,aunar esfuerzos tecnicos; administrativos y financieros para el diseno; desarrollo e implementacion de una...
1118,2026-01-24,DISTRITO ESPECIAL INDUSTRIAL Y PORTUARIO DE BARRANQUILLA,ZUMATI SAS,2000.0,prestacion de apoyo a la gestion para el desarrollo de un programa de capacitacion; y acompanamiento en el...


## Límites

- La búsqueda encuentra la frase "inteligencia artificial". Un contrato que solo dice "IA" o "machine learning" no aparece, así que las cifras son un piso.
- SECOP es lo que las entidades publican. Una entidad que no publica a tiempo no aparece.
- La clasificación por palabras es una aproximación: la muestra de arriba muestra dónde acierta y dónde no.
- 2026 llega al 3 de noviembre, y las entidades a veces publican con semanas de retraso: los últimos meses pueden crecer.

## Gráficas

In [21]:
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

SURFACE, TEXT, TEXT_2, MUTED, GRID = "#1a1a19", "#f0efec", "#c3c2b7", "#8a8980", "#2c2c2a"
S1, S2, NEUTRAL = "#3987e5", "#d95926", "#55544f"
plt.rcParams.update({"font.family": "Helvetica Neue", "font.size": 13, "axes.facecolor": SURFACE, "figure.facecolor": SURFACE,
                     "axes.edgecolor": SURFACE, "text.color": TEXT, "axes.labelcolor": TEXT_2, "xtick.color": TEXT_2, "ytick.color": TEXT_2})
SOURCE = "Fuente: SECOP I y II vía Croma. Contratos cuyo objeto dice \"inteligencia artificial\"."


def finish(fig, ax, title, subtitle, name, note=SOURCE):
    ax.spines[:].set_visible(False)
    fig.text(0.04, 0.95, title, fontsize=20, fontweight="bold", color=TEXT, ha="left", va="top")
    fig.text(0.04, 0.895, subtitle, fontsize=13, color=TEXT_2, ha="left", va="top")
    fig.text(0.04, 0.03, note, fontsize=10.5, color=MUTED, ha="left")
    fig.text(0.96, 0.03, "usecroma.com", fontsize=10.5, color=MUTED, ha="right")
    fig.savefig(CHARTS / name, dpi=200)
    plt.close(fig)

In [22]:
fig, ax = plt.subplots(figsize=(12, 6.75))
fig.subplots_adjust(left=0.07, right=0.96, top=0.8, bottom=0.14)
yrs = by_year.index.astype(str).tolist()
yrs[-1] = "2026\n(al 3 nov.)"
pub, cam = by_year["Entidades públicas"], by_year["Cámaras de comercio"]
ax.bar(yrs, pub, color=S1, width=0.62, label="Entidades públicas")
ax.bar(yrs, cam, bottom=pub, color=NEUTRAL, width=0.62, label="Cámaras de comercio")
for x, (p, t) in enumerate(zip(pub, by_year.Total)):
    ax.text(x, t + 12, f"{t:,}".replace(",", "."), ha="center", color=TEXT, fontsize=14, fontweight="bold")
    ax.text(x, p / 2, f"{p}", ha="center", va="center", color=SURFACE, fontsize=12, fontweight="bold")
ax.set_yticks([]); ax.tick_params(length=0)
ax.legend(frameon=False, loc="upper left", fontsize=12, labelcolor=TEXT_2)
finish(fig, ax, "Contratos en SECOP que mencionan la inteligencia artificial",
       "Por año de firma. Las cámaras de comercio publican en SECOP, pero son entidades privadas.", "01-contratos-por-ano.png")

In [23]:
m25 = monthly.loc["2025-01":"2025-12", "contratos"].to_numpy()
m26 = monthly.loc["2026-01":"2026-10", "contratos"].to_numpy()
months = ["ene", "feb", "mar", "abr", "may", "jun", "jul", "ago", "sep", "oct", "nov", "dic"]
fig, ax = plt.subplots(figsize=(12, 6.75))
fig.subplots_adjust(left=0.07, right=0.96, top=0.8, bottom=0.14)
ax.axvspan(0.97, 5.7, color="#26262a", zorder=0)
ax.text(3.33, max(m26) * 0.8, "Ley de Garantías\n31 ene a 21 jun 2026", ha="center", color=TEXT_2, fontsize=12)
ax.plot(range(12), m25, color=NEUTRAL, lw=2.5, marker="o", ms=5, label="2025")
ax.plot(range(len(m26)), m26, color=S2, lw=3, marker="o", ms=6, label="2026")
ax.annotate(f"{m26[0]} en enero", (0, m26[0]), xytext=(-0.25, m26[0] + 9), color=TEXT, fontsize=13, fontweight="bold", va="center")
ax.set_xticks(range(12), months); ax.tick_params(length=0)
ax.yaxis.grid(True, color=GRID); ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper right", fontsize=12, labelcolor=TEXT_2)
finish(fig, ax, "Enero de 2026: la carrera antes de la veda",
       "Contratos de entidades públicas que mencionan la IA, por mes de firma", "02-ley-de-garantias.png")

In [24]:
order_cats = mix.sort_values("total").index.tolist()
fig, ax = plt.subplots(figsize=(12, 6.75))
fig.subplots_adjust(left=0.3, right=0.94, top=0.8, bottom=0.14)
nat = mix.loc[order_cats, "persona natural"] if "persona natural" in mix else 0
org = mix.loc[order_cats, "organización"]
ax.barh(order_cats, nat, color=S1, height=0.6, label="Persona natural")
ax.barh(order_cats, org, left=nat, color=S2, height=0.6, label="Organización")
for y, t in enumerate(mix.loc[order_cats, "total"]):
    ax.text(t + 6, y, f"{t}", va="center", color=TEXT, fontsize=13, fontweight="bold")
ax.set_xticks([]); ax.tick_params(length=0)
ax.legend(frameon=False, loc="lower right", fontsize=12, labelcolor=TEXT_2)
finish(fig, ax, "Qué contrata el Estado cuando contrata IA",
       "Entidades públicas, 2025 y 2026, por lo que dice el objeto del contrato", "03-que-se-contrata.png")